# Figuras de la presentación al equipo

Deja en `../figures/` todo lo que `presentacion.tex` incluye: las cifras (`valores.tex`),
la tabla de Raven del pipeline, las figuras de matplotlib y los espectrogramas de ejemplo.
Nada se escribe a mano. Las fuentes:

- la comparación v3 (`runs/comparacion/comparacion_modelos_v3.json`): métricas globales y por clase;
- el volcado de prueba de `yolo26s_v3`: detecciones frente a anotaciones, con las mismas
  categorías que RE3.3;
- las tablas de Raven crudas de `data/raw/`, limpiadas en memoria con las reglas de RE1.1:
  cuántas anotaciones tiene cada tipo de llamada y por qué quedó fuera;
- el modelo sobre el audio, para los espectrogramas de ejemplo;
- dos cifras que este cuaderno no puede recalcular en local y lee de los `valores.tex` que ya
  las generaron: el costo de inferencia (capítulo 5, medido en GPU) y las discrepancias del
  corpus entero (RE3.3, con los pliegues del k-fold).

La presentación está en inglés: las cifras van con punto decimal y coma de miles.

In [ ]:
import json
import re
import subprocess
import sys
from dataclasses import replace
from functools import partial
from pathlib import Path
from typing import Any, NamedTuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import display
from matplotlib import font_manager, patheffects
from matplotlib.patches import Rectangle
from torchvision.ops import box_area, box_convert, box_iou

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR / "src"))

from core.config import MAX_DETECTIONS, RAW_DIR, P
from data.annotations import SPECIES_CODES, clean_annotations, list_files, species_of, unify_copies
from data.manifest import boxes_in_window
from data.species import CALL_TYPES, LabelSet, Species, make_label, split_label
from evaluation.evaluator import RawPredictions
from evaluation.metrics import (
    MATCH_IOU,
    Boxes,
    assignments,
    overlaps,
    rows_by_image,
    window_classes,
)
from evaluation.protocol import equalize
from models.registry import load_checkpoint
from prepare_annotations import COLUMNS, find_recording
from utils.audio import hz_to_y, load_clip, mel_spectrogram, mel_to_db, window_starts, y_to_hz
from utils.boxes import postprocess

FIG = Path.cwd().parent / "figures"
TAB = FIG / "tablas"
TAB.mkdir(parents=True, exist_ok=True)

RUN = "yolo26s_v3"  # el modelo seleccionado en el capítulo 5
RUN_DIR = PROJECT_DIR / "runs" / RUN
COMPARISON = PROJECT_DIR / "runs/comparacion/comparacion_modelos_v3.json"
TEST_DUMP = RUN_DIR / f"{RUN}_test_predictions.pt"
CH5_VALUES = PROJECT_DIR / "research/capitulos/05_oe2_detector/figures/valores.tex"
RE33_VALUES = PROJECT_DIR / "research/reportes/figures/RE_3-3/valores.tex"

# La paleta de presentacion.tex. Las cuatro categorías pasan el validador de paletas
# (banda de luminosidad, croma, separación con daltonismo y contraste sobre fondo claro).
INK, MUTED = "#17252B", "#5F6B70"
MATCH, BOX, LABEL, NONE = "#008A9E", "#C98419", "#7B5FB0", "#C64B36"
# Recall por tipo de llamada: una sola tinta, de oscuro (cubierta) a claro (débil). Son los
# nombres de los colores de presentacion.tex.
TIERS = ((0.80, "tierhigh"), (0.60, "tiermid"), (0.0, "tierlow"))
SLIDE_BG = "#F7F5EF"
# Ancho del texto de la diapositiva (16:9, márgenes de 8 mm): a este ancho, escala 1
TEXT_WIDTH_IN = 5.67

# Fira Sans es la letra de la presentación: las figuras la usan para que no se note el cambio.
FONTS = ["FiraSans-Regular", "FiraSans-Italic", "FiraSans-SemiBold", "FiraMono-Regular"]
kpse = subprocess.run(["kpsewhich", *(f"{f}.otf" for f in FONTS)], capture_output=True, text=True)
for font in kpse.stdout.splitlines():
    font_manager.fontManager.addfont(font)
mpl.rcParams.update(
    {
        "font.family": "Fira Sans",
        "font.size": 7,
        "text.color": INK,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "savefig.dpi": 450,
        "savefig.transparent": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

VALUES: dict[str, str] = {}  # las macros de valores.tex, sin la barra


def thousands(value: float) -> str:
    return f"{round(value):,}"


def tex(command: str, *args: object) -> str:
    # tex("callrow", "sqc", "0.99") -> \callrow{sqc}{0.99}
    return f"\\{command}" + "".join(f"{{{arg}}}" for arg in args)


def write(path: Path, body: str) -> None:
    path.write_text("% generado por notebook/figuras_presentacion.ipynb; no editar\n" + body)


def two_rows(items: list[str], first: int, gap: str) -> str:
    # Lado a lado, en dos filas: las `first` primeras arriba
    row = "%\n\\hfill\n".join
    return row(items[:first]) + f"%\n\\par\\vspace{{{gap}}}\n" + row(items[first:]) + "%\n"


def save(fig, name: str, tight: bool = True) -> None:
    # Sin `tight`, la imagen sale exactamente del tamaño de la figura: así todas las de una
    # grilla miden lo mismo
    fig.savefig(FIG / name, bbox_inches="tight" if tight else None, pad_inches=0.01)
    plt.show()

## Métricas globales del modelo

In [ ]:
comparison = json.loads(COMPARISON.read_text())
model_row = next(m for m in comparison["models"] if m["model"] == RUN)
per_class = pd.DataFrame(model_row["per_class"]).set_index("name")


def read_macros(path: Path) -> dict[str, str]:
    return dict(re.findall(r"\\newcommand\{\\(\w+)\}\{([^}]*)\}", path.read_text()))


ch5 = read_macros(CH5_VALUES)
VALUES.update(
    threshold=f"{model_row['threshold']:.2f}",
    recall=f"{model_row['test']['recall']:.2f}",
    precision=f"{model_row['test']['precision']:.2f}",
    map_thirty=f"{model_row['map_30']:.2f}",
    species_correct=f"{100 * model_row['axes']['species_correct']:.0f}",
    call_correct=f"{100 * model_row['axes']['class_correct']:.0f}",
    test_boxes=thousands(comparison["test"]["boxes"]),
    test_recordings=thousands(comparison["test"]["recordings"]),
    n_models=str(len(comparison["models"])),
    n_classes=str(len(per_class)),
    # Medido en el capítulo 5 sobre GPU; aquí no hay una para repetirlo
    realtime=ch5["rRealtimeYol"],
    gpu_name=ch5["rGpuName"],
)
{
    k: VALUES[k]
    for k in ("threshold", "recall", "precision", "map_thirty", "species_correct", "realtime")
}

## Anotaciones por tipo de llamada y qué quedó fuera

Las tablas crudas se limpian en memoria igual que `prepare_annotations.py` (sin escribir
`data/cleaned/`). Las reglas con que se armó el conjunto del modelo (mínimo de anotaciones,
frases excluidas, tipos unidos) se leen del `config.json` de la corrida, no del código de hoy.

In [ ]:
dataset = json.loads((RUN_DIR / "config.json").read_text())["dataset"]
MIN_COUNT = int(dataset["min_pair_count"])
PHRASES = set(dataset["excluded_labels"])
JOINED = dataset["joined_labels"]

frames: list[pd.DataFrame] = []
for table in list_files(RAW_DIR, ".txt"):
    audio = find_recording(table) if species_of(table) in SPECIES_CODES else None
    if audio is None:
        continue
    try:
        frame = clean_annotations(pd.read_csv(table, sep="\t"), species_of(audio))
    except Exception:  # una tabla ilegible no detiene nada, igual que en el pipeline
        continue
    if len(frame):
        frames.append(frame[COLUMNS].assign(audio_path=str(audio)))

In [ ]:
annotations = unify_copies(pd.concat(frames, ignore_index=True))
annotations["requires_review"] = annotations["requires_review"].astype(bool)
annotations["label"] = (annotations["species"] + "/" + annotations["call_type"].fillna("")).replace(
    JOINED
)
eligible = annotations[~annotations["requires_review"]]
n_all = annotations["label"].value_counts()
n_eligible = eligible["label"].value_counts()

# Un renglón por tipo de llamada del vocabulario (los unidos, una sola vez) y por frase excluida.
vocabulary = []
for species, codes in CALL_TYPES.items():
    code = species.name.lower()
    names = [JOINED.get(make_label(code, c), make_label(code, c)) for c in codes]
    names += sorted(p for p in PHRASES if split_label(p)[0] == code)
    for name in dict.fromkeys(names):
        if name in per_class.index:
            status = "trained"
        elif name in PHRASES:
            status = "phrase"
        elif name in n_eligible:
            status = "few"
        else:
            continue  # sin ninguna anotación usable: no es un tipo de llamada del corpus
        vocabulary.append(
            {
                "species": code,
                "label": name,
                "call": split_label(name)[1],
                "status": status,
                "annotations": int(n_all.get(name, 0)),
                "eligible": int(n_eligible.get(name, 0)),
            }
        )
calls = (
    pd.DataFrame(vocabulary)
    .set_index("label")
    .join(per_class[["n_gt", "recall", "precision", "ap", "window_class"]])
)
assert set(per_class.index) <= set(calls.index), "toda clase del modelo es un tipo del vocabulario"
assert (calls.loc[calls.status == "few", "eligible"] < MIN_COUNT).all()

cc = calls.loc["cc/cc"]
VALUES.update(
    min_count=str(MIN_COUNT),
    n_trained=str(int((calls.status == "trained").sum())),
    n_not_trained=str(int((calls.status != "trained").sum())),
    cc_annotations=str(cc.annotations),
    cc_recordings=str(eligible.loc[eligible.label == "cc/cc", "audio_path"].nunique()),
    cc_missing=str(MIN_COUNT - cc.annotations),
)
calls.loc[calls.status != "trained", ["status", "annotations"]].sort_values("annotations")

## Desglose por especie y tipo de llamada

Una tarjeta por especie, todas del mismo tamaño, en dos filas de cuatro, de mejor a peor
recall de la especie (ponderado por sus cajas de prueba); *Cebus*, sin ningún tipo
entrenado, al final. En cada tarjeta, los tipos entrenados por recall y, al pie, los que
quedaron fuera con su cantidad de anotaciones. Se escriben como llamadas a `\speciescard`,
`\callrow` y `\nottrained`, que dibuja `presentacion.tex`: el texto queda en la letra de
la presentación y los bordes de las ocho tarjetas, alineados.

In [ ]:
SPECIES_NAMES = {
    "aa": "A. azarae",
    "ac": "A. chamek",
    "as": "A. sara",
    "cc": "C. cuscinus",
    "lw": "L. weddelli",
    "pt": "P. toppini",
    "sb": "S. boliviensis",
    "sm": "S. macrocephalus",
}
assert set(SPECIES_NAMES) == {s.name.lower() for s in Species}

# El tono va por la cifra que se muestra, redondeada a dos decimales: un 0,795 se lee «0.80» y
# cuenta como cubierta
trained = calls[calls.status == "trained"].assign(
    tier=lambda d: [next(tier for floor, tier in TIERS if r >= floor) for r in d.recall.round(2)]
)
left_out = calls[calls.status != "trained"]
species_recall = {  # ponderado por las cajas de prueba de cada tipo
    code: (g.recall * g.n_gt).sum() / g.n_gt.sum() for code, g in trained.groupby("species")
}
order = sorted(SPECIES_NAMES, key=lambda c: -species_recall.get(c, -1.0))

cards = []
for code in order:
    name = SPECIES_NAMES[code].replace(". ", ".~")
    kept = trained[trained.species == code].sort_values("recall", ascending=False)
    dropped = left_out[left_out.species == code].sort_values("annotations", ascending=False)
    if kept.empty:
        n_recordings = eligible.loc[eligible.species == code, "audio_path"].nunique()
        cards.append(tex("emptycard", code.upper(), name, dropped.annotations.sum(), n_recordings))
        continue
    rows = []
    for r in kept.itertuples():
        call = f"{r.call}*" if r.window_class else r.call  # *: clase de ventana
        value = f"{r.recall:.2f}"
        if r.tier == "tierhigh":
            value = tex("textbf", value)
        rows.append(
            "  " + tex("callrow", call, f"{r.recall:.3f}", f"{r.precision:.3f}", r.tier, value)
        )
    footer = " \\textperiodcentered{} ".join(
        # el número no se separa de su código
        tex("code", r.call) + "~" + ("phrase" if r.status == "phrase" else str(r.annotations))
        for r in dropped.itertuples()
    )
    recall, n_boxes = f"{species_recall[code]:.2f}", thousands(kept.n_gt.sum())
    card = tex("speciescard", code.upper(), name, recall, n_boxes, "%\n" + "\n".join(rows))
    cards.append(card + "%\n  " + ("{" + tex("nottrained", footer) + "}" if footer else "{}"))
write(TAB / "especies.tex", two_rows(cards, 4, "2mm"))
print([c.upper() for c in order])

covered = trained[trained.tier == "tierhigh"]
VALUES.update(
    n_covered=str(len(covered)),
    covered_share=f"{100 * covered.n_gt.sum() / trained.n_gt.sum():.0f}",
)
{k: VALUES[k] for k in ("n_trained", "n_not_trained", "n_covered", "covered_share")}

## Detecciones frente a las anotaciones entregadas

Sobre el volcado de prueba, al umbral de operación, con la regla de RE3.3: cada detección que
no acierta cae en una sola categoría, en este orden. **Caja distinta** si se superpone con una
anotación de su clase pero no la acierta (IoU < 0,3, o la anotación ya la tomó otra
detección, o cubre varias); **otra llamada** si encuadra (IoU ≥ 0,3) una anotación de otra
clase; **nada anotado** si no toca ninguna. RE3.3 separa las dos primeras formas de «caja
distinta» (encuadre y unidad); aquí van juntas. Las dos clases de ventana (`as/hc`, `pt/dc`),
cuya caja ocupa la ventana entera, quedan fuera, como en RE3.3.

In [ ]:
CATEGORIES = {"match": MATCH, "box": BOX, "label": LABEL, "none": NONE}
COVER = 0.5  # una detección "cubre" una anotación si tapa al menos esta fracción de su área

dump = RawPredictions.load(TEST_DUMP)
labels = LabelSet(dump.labels)
THRESHOLD = model_row["threshold"]


def miss(
    p: torch.Tensor, label: int, t: torch.Tensor, t_labels: torch.Tensor, t_claimed: torch.Tensor
) -> tuple[str, int | None]:
    # La regla de RE3.3 para una detección que no acertó, frente a las anotaciones de su ventana
    # (cajas xyxy): su categoría y la anotación a que se refiere
    if not len(t):
        return "none", None
    iou = box_iou(p[None], t)[0]
    same = t_labels == label
    inter = (torch.min(p[2:], t[:, 2:]) - torch.max(p[:2], t[:, :2])).clamp(min=0).prod(dim=1)
    covered = same & (inter / box_area(t).clamp(min=1e-9) >= COVER)
    best_same = int((iou * same).argmax()) if same.any() else None
    best_other = int((iou * ~same).argmax()) if (~same).any() else None
    if best_same is not None and iou[best_same] >= MATCH_IOU and t_claimed[best_same]:
        return "box", best_same
    if best_other is not None and iou[best_other] >= MATCH_IOU:
        return "label", best_other
    if int(covered.sum()) >= 2 or (best_same is not None and iou[best_same] > 0):
        return "box", best_same
    return "none", None


def box_frame(boxes: torch.Tensor, ids: torch.Tensor) -> pd.DataFrame:
    # Una caja (cxcywh) por fila: sus bordes en segundos de la ventana y hercios, redondeados,
    # para leerla y elegir el recorte del dibujo; esquina y tamaño, para dibujarla
    xyxy = box_convert(boxes, "cxcywh", "xyxy").numpy()
    cx, cy, w, h = boxes.double().T.numpy()
    return pd.DataFrame(
        {
            "label": [labels.name(i) for i in ids.tolist()],
            "t0": (xyxy[:, 0] * P.clip_len_s).round(2),
            "t1": (xyxy[:, 2] * P.clip_len_s).round(2),
            "f0": y_to_hz(xyxy[:, 1], P).round(),
            "f1": y_to_hz(xyxy[:, 3], P).round(),
            "x0": (cx - w / 2) * P.clip_len_s,
            "y0": cy - h / 2,
            "width": w * P.clip_len_s,
            "height": h,
        }
    )


def categorize(pred: Boxes, truth: Boxes) -> pd.DataFrame:
    # Una detección por fila, con su categoría. Acierta si toma una anotación de su clase con
    # IoU ≥ 0,3, en orden de score, como `hits`; si no, va por la regla de `miss`.
    pairs = assignments(overlaps(pred, truth, class_aware=True), MATCH_IOU)
    found = {p for p, _ in pairs}
    claimed = torch.zeros(len(truth.boxes), dtype=torch.bool)
    claimed[[t for _, t in pairs]] = True
    p_xyxy = box_convert(pred.boxes, "cxcywh", "xyxy")
    t_xyxy = box_convert(truth.boxes, "cxcywh", "xyxy")
    by_window = rows_by_image(truth.image_ids)
    category, annotated_as = [], []
    for row, (label, window) in enumerate(zip(pred.labels.tolist(), pred.image_ids.tolist())):
        t = torch.tensor(by_window.get(window, []), dtype=torch.long)
        c, k = (
            ("match", None)
            if row in found
            else miss(p_xyxy[row], label, t_xyxy[t], truth.labels[t], claimed[t])
        )
        category.append(c)
        annotated_as.append(None if k is None else labels.name(int(truth.labels[t[k]])))
    return box_frame(pred.boxes, pred.labels).assign(
        species=lambda d: [split_label(name)[0] for name in d.label],
        score=pred.scores.tolist(),
        category=category,
        annotated_as=annotated_as,
        window=pred.image_ids.tolist(),
    )


predictions = equalize(dump.predictions)
truth = dump.truth
window_labels = {labels.name(c) for c in window_classes(truth, len(labels))}
assert window_labels == set(per_class.index[per_class.window_class])
detections = categorize(predictions.select(predictions.scores >= THRESHOLD), truth)
detections = detections[~detections.label.isin(window_labels)].assign(
    recording=lambda d: [Path(dump.recording_names[int(dump.recordings[w])]).name for w in d.window]
)
counts = detections.category.value_counts().reindex(list(CATEGORIES), fill_value=0)

# El otro lado: de las anotaciones (sin las clases de ventana), cuántas tomó una detección. Cada
# acierto toma exactamente una.
n_annotations = sum(labels.name(c) not in window_labels for c in truth.labels.tolist())
n_found = int(counts["match"])

mislabeled = detections[detections.category == "label"]
same_species = (mislabeled.species == [split_label(a)[0] for a in mislabeled.annotated_as]).mean()
pairs = (mislabeled.annotated_as + " → " + mislabeled.label).value_counts()
n_detections = len(detections)
VALUES.update(
    n_detections=thousands(n_detections),
    n_annotations=thousands(n_annotations),
    n_missed=thousands(n_annotations - n_found),
    pct_found=f"{100 * n_found / n_annotations:.0f}",
    pct_missed=f"{100 * (n_annotations - n_found) / n_annotations:.0f}",
    label_same_species=f"{100 * same_species:.0f}",
    top_confusion=pairs.index[0].replace("→", "$\\to$"),
    top_confusion_n=str(pairs.iloc[0]),
)
for category, n in counts.items():
    VALUES.update(
        {f"n_{category}": thousands(n), f"pct_{category}": f"{100 * n / n_detections:.0f}"}
    )
print(counts.to_dict(), n_annotations, n_found)
pairs.head(6)

El corpus entero (RE3.3, con predicciones fuera de muestra en train por k-fold) no se puede
recalcular aquí: los pliegues no están en local. Se leen sus cifras.

In [ ]:
re33 = read_macros(RE33_VALUES)
VALUES.update(  # `11\,864` → `11,864`
    corpus_none=re33["nD"].replace("\\,", ","),
    corpus_none_half=re33["nHalf"].replace("\\,", ","),
    corpus_none_per_hour=re33["dPerHour"].split(",")[0],
)

In [ ]:
MISSED = "#C9C1AF"
NAMES = {
    "match": "matched",
    "box": "different box",
    "label": "different call",
    "none": "nothing annotated",
    "missed": "missed",
}
# Los segmentos angostos se rotulan encima de la barra, hacia afuera de su vecino
OUTSIDE = {"box": "right", "label": "left"}
bars = {
    "Detections": [(c, counts[c], CATEGORIES[c]) for c in CATEGORIES],
    "Annotations": [("match", n_found, MATCH), ("missed", n_annotations - n_found, MISSED)],
}
scale = max(n_detections, n_annotations)
fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 0.85))
for y, (name, parts) in enumerate(bars.items()):
    left = 0
    for key, n, color in parts:
        x0, x1 = left / scale, (left + n) / scale
        middle = (x0 + x1) / 2
        ax.barh(y, x1 - x0, left=x0, height=0.62, color=color, edgecolor=SLIDE_BG, linewidth=0.8)
        text = f"{thousands(n)} {NAMES[key]}"
        if key in OUTSIDE:
            ax.plot([middle, middle], [y - 0.33, y - 0.55], color=color, lw=0.6)
            ax.text(
                middle,
                y - 0.6,
                text,
                ha=OUTSIDE[key],
                va="bottom",
                fontsize=6.5,
                color=color,
                fontweight="semibold",
            )
        else:
            ax.text(
                middle,
                y,
                text,
                ha="center",
                va="center",
                color=INK if color == MISSED else "white",
                fontweight="semibold",
            )
        left += n
    ax.text(-0.01, y, name, ha="right", va="center", fontsize=7.5, fontweight="semibold")
    ax.text(left / scale + 0.008, y, thousands(left), ha="left", va="center", color=MUTED)
ax.set(xlim=(0, 1.07), ylim=(1.45, -1.0))
ax.axis("off")
save(fig, "contraste.pdf")

## Espectrogramas: el modelo sobre el audio

El caché de `data/processed/` que hay en local no es el de la corrida v3 (tiene otra
partición), así que las ventanas de ejemplo no se toman del caché: se cargan del audio y se
pasan por el modelo, como hace `detect.py`. Las anotaciones se recortan a la ventana con la
misma regla que el caché (`boxes_in_window`) y cada detección se clasifica con
`categorize`, igual que en el volcado. Las grabaciones son todas de prueba.

In [ ]:
loaded = load_checkpoint(RUN_DIR / "best.pt")
detector = loaded.model
assert loaded.labels.names == labels.names
to_mel = mel_spectrogram()
# Sólo para dibujar: el mismo log-mel (misma escala y mismo rango de frecuencias, así que las
# cajas caen en el mismo lugar) con tres veces más bandas y cuatro veces más cuadros. El modelo
# sigue viendo su entrada de 128 bandas.
to_display_mel = mel_spectrogram(replace(P, n_fft=8192, hop_length=100, n_mels=384))
test_recordings = {Path(name).name for name in dump.recording_names}
experiment = eligible[eligible.label.isin(labels.names)]


class View(NamedTuple):  # una ventana: lo que el modelo detectó y lo anotado
    path: str
    start: float
    det: pd.DataFrame
    truth: pd.DataFrame


def windows(name: str) -> list[View]:
    # Todas las ventanas de una grabación de prueba
    assert name in test_recordings, f"{name} no es de prueba"
    path = next(p for p in experiment.audio_path if Path(p).name == name)
    group = experiment[experiment.audio_path == path]
    class_ids = np.array([labels.id(label) for label in group.label])
    views = []
    for start in window_starts(sf.info(path).duration, P).tolist():
        mel = to_mel(load_clip(path, start))[None, None]
        det = postprocess(detector.detect(mel, THRESHOLD)[0], detector.nms_iou, MAX_DETECTIONS)
        boxes, ids, _ = boxes_in_window(group, class_ids, start, P)
        target = {"boxes": torch.tensor(boxes, dtype=torch.float32), "labels": torch.tensor(ids)}
        truth = Boxes.truth(target, 0)
        det = categorize(Boxes.of(det, 0), truth)
        views.append(View(path, start, det, box_frame(truth.boxes, truth.labels)))
    return views


def best_window(name: str, wanted, key) -> View:
    # La mejor ventana de la grabación según `key`, entre las que cumplen `wanted`
    views = [view for view in windows(name) if wanted(view)]
    assert views, f"ninguna ventana de {name} cumple la condición"
    return max(views, key=key)


def has(view: View, category: str, label: str, annotated: str | None = None) -> bool:
    # Si tiene una detección de esa categoría y etiqueta (y anotada como `annotated`)
    det = view.det
    hit = (det.category == category) & (det.label == label)
    if annotated is not None:
        hit &= det.annotated_as == annotated
    return bool(hit.any())


def top(view: View, category: str) -> float:
    # El score de su mejor detección de esa categoría
    return float(view.det.score[view.det.category == category].max())

In [ ]:
# Las cajas llevan un borde de contraste debajo, para que se lean sobre el fondo oscuro y sobre la
# energía clara de inferno: las detecciones, en línea continua; las anotaciones, cortada.
DETECTION: dict[str, Any] = {
    "fill": False,
    "lw": 1.3,
    "path_effects": [patheffects.withStroke(linewidth=2.6, foreground="white")],
}
ANNOTATION: dict[str, Any] = {
    "fill": False,
    "ec": "white",
    "lw": 1.0,
    "ls": (0, (2.5, 1.5)),
    "path_effects": [patheffects.withStroke(linewidth=2.2, foreground=INK)],
}
TAG: dict[str, Any] = {  # el rótulo de una caja
    "fontsize": 6.5,
    "family": "Fira Mono",
    "bbox": {"facecolor": "white", "edgecolor": "none", "pad": 0.8, "alpha": 0.9},
    "clip_on": True,
}
WHOLE = (0.0, P.clip_len_s, P.f_min, P.f_max)  # un recorte: segundos de la ventana y hercios


def anchor(x0: float, x1: float, t0: float, t1: float) -> tuple[float, str]:
    # El rótulo de una caja va desde su borde izquierdo; si la caja está en el tercio derecho
    # del recorte, termina en su borde derecho, para no salirse del dibujo
    if x0 - t0 > 0.6 * (t1 - t0):
        return min(x1, t1 - 0.01), "right"
    return max(x0, t0 + 0.01), "left"


def draw(
    ax,
    view: View,
    det: pd.DataFrame | None = None,
    crop: tuple[float, float, float, float] = WHOLE,
    truth: bool = False,
    text: bool = True,
    truth_text: bool = True,
) -> None:
    # `det`: las detecciones que se dibujan, del color de su categoría; `truth`: también las
    # anotaciones
    t0, t1, fmin, fmax = crop
    mel = mel_to_db(to_display_mel(load_clip(view.path, view.start))).numpy()
    low, high = np.percentile(mel, [20, 99.8])  # el fondo, oscuro; la energía, clara
    y_low, y_high = (float(v) for v in hz_to_y(np.array([fmin, fmax]), P))
    ax.imshow(
        mel,
        origin="lower",
        aspect="auto",
        cmap="inferno",
        vmin=low,
        vmax=high,
        extent=(0, P.clip_len_s, 0, 1),
        interpolation="bilinear",
    )
    ax.set(xlim=(t0, t1), ylim=(y_low, y_high), xticks=[], yticks=[])
    for spine in ax.spines.values():
        spine.set(visible=True, color="#C9C1AF", linewidth=0.5)
    if truth:
        b = view.truth[(view.truth.x0 <= t1) & (view.truth.x0 + view.truth.width >= t0)]
        for x0, y0, w, h, label in zip(b.x0, b.y0, b.width, b.height, b.label):
            ax.add_patch(Rectangle((x0, y0), w, h, **ANNOTATION))
            if text and truth_text:
                x, ha = anchor(x0, x0 + w, t0, t1)
                ax.text(x, y0 - 0.006, label, va="top", ha=ha, color=INK, **TAG)
    if det is None:
        return
    b = det[(det.x0 <= t1) & (det.x0 + det.width >= t0)]
    for x0, y0, w, h, label, score, category in zip(
        b.x0, b.y0, b.width, b.height, b.label, b.score, b.category
    ):
        color = CATEGORIES[category]
        ax.add_patch(Rectangle((x0, y0), w, h, ec=color, **DETECTION))
        if text:
            x, ha = anchor(x0, x0 + w, t0, t1)
            # Encima de la caja; si la caja llega al borde de arriba, adentro
            if y0 + h > y_high - 0.12 * (y_high - y_low):
                y, va = min(y0 + h, y_high - 0.004) - 0.01 * (y_high - y_low), "top"
            else:
                y, va = y0 + h + 0.006, "bottom"
            ax.text(x, y, f"{label} {score:.2f}", va=va, ha=ha, color=color, **TAG)

### La ventana del pipeline

Una ventana de prueba con dos especies (*Saimiri* y *Sapajus*) en que todas las anotaciones
tienen su acierto y no sobra ninguna detección: la buscamos en el volcado entre las ventanas
con dos especies anotadas y todas sus anotaciones tomadas.

In [ ]:
def all_matched(view: View) -> bool:
    # Cuatro detecciones o más, de dos especies o más, todas aciertos, y ninguna anotación sin
    # el suyo
    det = view.det
    return (
        len(det) >= 4
        and bool((det.category == "match").all())
        and det.species.nunique() >= 2
        and len(view.truth) == len(det)
    )


pipeline = best_window("20240522_071743.wav", all_matched, lambda view: len(view.det))
print(pipeline.start)
pipeline.det

In [ ]:
fig, ax = plt.subplots(figsize=(1.3, 0.95))
draw(ax, pipeline)
save(fig, "pipeline_espectrograma.png")
fig, ax = plt.subplots(figsize=(1.3, 0.95))
draw(ax, pipeline, pipeline.det, text=False)
save(fig, "pipeline_detecciones.png")

# La grabación alrededor de la ventana, con tres ventanas consecutivas encima.
start = pipeline.start
context = (max(start - 3.0, 0.0), start + 6.0)
with sf.SoundFile(pipeline.path) as handle:
    rate = handle.samplerate
    handle.seek(int(context[0] * rate))
    wave = handle.read(int((context[1] - context[0]) * rate), always_2d=True).mean(1)
t = context[0] + np.arange(len(wave)) / rate
step = max(len(wave) // 4000, 1)
fig, ax = plt.subplots(figsize=(1.5, 0.95))
ax.plot(
    t[::step], np.clip(wave[::step] / np.percentile(np.abs(wave), 99.9), -1, 1), color=MUTED, lw=0.3
)
for k, offset in enumerate((-P.clip_hop_s, 0.0, P.clip_hop_s)):
    ax.plot(
        [start + offset, start + offset + P.clip_len_s],
        [1.25 + 0.22 * k] * 2,
        color=MATCH if offset == 0 else "#B9B2A3",
        lw=1.6 if offset == 0 else 1.0,
        solid_capstyle="butt",
    )
ax.set(xlim=context, ylim=(-1.05, 1.85))
ax.axis("off")
save(fig, "pipeline_audio.pdf")

# La tabla de Raven que saldría de esa ventana, en tiempo de la grabación.
det = pipeline.det
raven = pd.DataFrame(
    {
        "begin": start + det.x0,
        "end": start + det.x0 + det.width,
        "low": y_to_hz(det.y0.to_numpy(), P),
        "high": y_to_hz((det.y0 + det.height).to_numpy(), P),
        "species": det.species.str.upper(),
        "call": [split_label(label)[1].upper() for label in det.label],
        "score": det.score,
    }
).sort_values("begin")
write(
    TAB / "raven.tex",
    " \\\\\n".join(
        f"{r.begin:.2f} & {r.end:.2f} & {r.low:,.0f} & {r.high:,.0f} & {r.species} & {r.call} & "
        f"{r.score:.2f}"
        for r in raven.itertuples()
    )
    + "\n",
)
raven

### Un ejemplo por cada tipo de diferencia

Elegidas entre las discrepancias de mayor score del volcado de prueba, una grabación por
tipo, buscando la que mejor se lee:

- **caja distinta**: sílabas de comida de *Sapajus* (`sm/fs`) que el equipo encuadró
  ajustadas y el modelo, altas: las dos alturas de caja de esa etiqueta;
- **nada anotado**: dos alarmas terrestres de *Leontocebus* (`lw/ta`) casi iguales, una anotada
  y la otra no; la anotada sirve de referencia de acierto;
- **otra llamada**: un trino (`lw/trino`) que el modelo llama `lw/vc`, la confusión entre
  llamadas parecidas de la misma especie.

In [ ]:
# Por categoría: la grabación, qué tiene que tener la ventana, el recorte del dibujo (segundos de
# la ventana y hercios, leídos de su tabla) y si lleva rótulos.
EXAMPLES = {
    "box": (
        "20240916_102432.wav",
        lambda v: has(v, "box", "sm/fs"),
        (0.0, 0.85, 900, 5500),
        False,
    ),
    "none": (
        "20240129_122526.wav",
        lambda v: has(v, "none", "lw/ta") and has(v, "match", "lw/ta"),
        (0.9, 3.0, 1200, 12000),
        True,
    ),
    "label": (
        "20240520_070831.wav",
        lambda v: has(v, "label", "lw/vc", annotated="lw/trino"),
        (0.85, 1.85, 4500, 12000),
        True,
    ),
}
for category, (name, wanted, crop, text) in EXAMPLES.items():
    view = best_window(name, wanted, partial(top, category=category))
    fig, ax = plt.subplots(figsize=(1.8, 1.05))
    det = view.det[view.det.category.isin([category, "match"])]
    draw(ax, view, det, crop, truth=True, text=text)
    save(fig, f"ejemplo_{category}.png")
    print(category, view.start)
    display(view.det)

## Apéndice: las cuatro arquitecturas entrenadas

La misma comparación del capítulo 5: cada modelo en su umbral de validación, medido en
prueba. El intervalo de recall (bootstrap por grabación), el costo en GPU y los parámetros
los calculó el cuaderno del capítulo 5; aquí se leen de sus macros.

In [ ]:
ARCHITECTURES = {  # corrida -> (nombre, sufijo de sus macros en el capítulo 5, tipo, preentrenamiento)
    "detr_t10_logmel_birds_v3": ("AST-Deformable-DETR", "Det", "Transformer, set head", "AudioSet"),
    "frcnn_v3": ("Faster R-CNN R50-FPN", "Frc", "Two-stage CNN", "COCO"),
    "rtdetr_l_v3": ("RT-DETR-L", "Rtd", "CNN, set head", "COCO"),
    "yolo26s_v3": ("YOLO26s", "Yol", "One-stage CNN", "COCO"),
}
CI_AXIS = (0.74, 0.86)  # el eje de los intervalos, igual para las cuatro filas

lines = []
for row in sorted(comparison["models"], key=lambda m: -m["map_30"]):
    name, suffix, kind, pretrained = ARCHITECTURES[row["model"]]
    # `[0,787; 0,835]`, con coma decimal
    low, high = (
        float(v.replace(",", ".")) for v in ch5[f"rRecallCi{suffix}"].strip("[]").split(";")
    )
    recall = row["test"]["recall"]
    line = " & ".join(
        [
            tex("archcell", name, kind),
            pretrained,
            ch5[f"cParams{suffix}"].replace(",", "."),
            tex("ciplot", f"{low:.3f}", f"{recall:.3f}", f"{high:.3f}"),
            f"{recall:.2f}",
            f"{row['test']['precision']:.2f}",
            f"{row['map_30']:.2f}",
            ch5[f"rGpuMin{suffix}"].replace(",", "."),
        ]
    )
    # El modelo seleccionado, resaltado
    lines.append(tex("rowcolor", "match!12") + line if row["model"] == RUN else line)
write(TAB / "modelos.tex", " \\\\\n".join(lines) + "\n")
VALUES.update(
    ci_low=f"{CI_AXIS[0]:.2f}",
    ci_high=f"{CI_AXIS[1]:.2f}",
    repo_url=ch5["rRepoUrl"].removeprefix("https://"),
)
print((TAB / "modelos.tex").read_text())

## Apéndice: más detecciones donde no hay nada anotado

De cada especie, la detección «nada anotado» de mayor score en prueba, en una grabación
distinta de la del ejemplo principal. Se busca su ventana en el audio y se recorta alrededor
de la caja. El pie lleva la grabación y el segundo, para que el equipo la abra en Raven.

In [ ]:
shown = {name for name, *_ in EXAMPLES.values()}
candidates = (
    detections[(detections.category == "none") & ~detections.recording.isin(shown)]
    .sort_values("score", ascending=False)
    .drop_duplicates("species")
    .head(6)
)
captions = []
for k, (recording, label) in enumerate(zip(candidates.recording, candidates.label), start=1):
    view = best_window(
        recording, partial(has, category="none", label=label), partial(top, category="none")
    )
    target = view.det.index[(view.det.category == "none") & (view.det.label == label)][0]
    box = view.det.loc[target]
    crop = (
        max(box.t0 - 0.7, 0.0),
        min(box.t1 + 0.7, P.clip_len_s),
        max(box.f0 / 2.0, P.f_min),
        min(box.f1 * 1.8, P.f_max),
    )
    fig = plt.figure(figsize=(1.8, 0.95))
    ax = fig.add_axes((0, 0, 1, 1))
    # Sólo esa detección
    draw(ax, view, view.det.loc[[target]], crop, truth=True, truth_text=False)
    save(fig, f"sin_anotacion_{k}.png", tight=False)
    # El pie: la grabación y el segundo, para abrirla en Raven
    stem, second = Path(recording).stem.replace("_", "\\_"), f"{view.start + box.t0:.1f}"
    captions.append(
        tex("moreexample", f"sin_anotacion_{k}", label, f"{box.score:.2f}", stem, second)
    )
write(TAB / "sin_anotacion.tex", two_rows(captions, 3, "2.5mm"))
candidates[["recording", "species", "score"]]

In [ ]:
# map_thirty -> \mapThirty
lines = [
    tex("newcommand", "\\" + re.sub(r"_([a-z])", lambda m: m[1].upper(), k), v)
    for k, v in VALUES.items()
]
write(FIG / "valores.tex", "\n".join(lines) + "\n")
pd.Series(VALUES)